In [ ]:
import gc
import torch
from pathlib import Path
from anomalib.data import MVTecAD
from anomalib.models import Patchcore
from anomalib.engine import Engine

# Constants
DATASET_PATH = Path("./datasets")
ALL_CATEGORIES = [f.name for f in DATASET_PATH.iterdir() if f.is_dir()]

In [ ]:
# Init dictionnary to store metrics for each category
metrics_dict = {}
for category in ALL_CATEGORIES:
    model = None
    datamodule = None
    engine = None
    try:
        # Initialize the model, datamodule, and engine for the current category
        model = Patchcore(backbone="resnet18", pre_trained=True)
        datamodule = MVTecAD(
            root=DATASET_PATH,
            category=category,
            train_batch_size=16,
            eval_batch_size=16,
            num_workers=0,
        )
        engine = Engine(
            accelerator="auto",
            default_root_dir=f"./results/{category}",
            enable_model_summary=False,
            enable_progress_bar=False,
        )
        # Training
        print(f"Training for '{category}' images...")
        engine.fit(model=model, datamodule=datamodule)
        # Evaluation
        print("Evaluating model performance...")
        test_metrics = engine.test(model=model, datamodule=datamodule, verbose=False)
        metrics_dict[category] = test_metrics[0] if isinstance(test_metrics, list) else test_metrics

    finally:
        # Free memory for next iteration
        del model, datamodule, engine
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [4]:
for category, metrics in metrics_dict.items():
    print(f"Test Metrics for '{category}':\n {metrics}")

Test Metrics for 'bottle':
 {'image_AUROC': 1.0, 'image_F1Score': 0.9919999837875366, 'pixel_AUROC': 0.9784282445907593, 'pixel_F1Score': 0.6691327691078186}
Test Metrics for 'cable':
 {'image_AUROC': 0.9748875498771667, 'image_F1Score': 0.9304812550544739, 'pixel_AUROC': 0.9818689227104187, 'pixel_F1Score': 0.6190202832221985}
Test Metrics for 'capsule':
 {'image_AUROC': 0.9816514253616333, 'image_F1Score': 0.9769585132598877, 'pixel_AUROC': 0.9881346225738525, 'pixel_F1Score': 0.4899943470954895}
Test Metrics for 'carpet':
 {'image_AUROC': 0.9799357652664185, 'image_F1Score': 0.9608938694000244, 'pixel_AUROC': 0.9879415035247803, 'pixel_F1Score': 0.5637587904930115}
Test Metrics for 'grid':
 {'image_AUROC': 0.9515455365180969, 'image_F1Score': 0.9322034120559692, 'pixel_AUROC': 0.970248818397522, 'pixel_F1Score': 0.3567342162132263}
Test Metrics for 'hazelnut':
 {'image_AUROC': 0.9989286661148071, 'image_F1Score': 0.978723406791687, 'pixel_AUROC': 0.9891660213470459, 'pixel_F1Score':